In [1]:
import pandas as pd
import networkx as nx
from sklearn.model_selection import GroupShuffleSplit

# 1. Load your dataset
df = pd.read_csv("../preprocessing/train_data_with_preprocessing2.csv")

# 2. Build the Bipartite Graph
G = nx.Graph()

# Add prefixes to avoid collisions between identical context and reply strings
df['ctx_node'] = "CTX_" + df['prev_clean'].astype(str)
df['rep_node'] = "REP_" + df['target_clean'].astype(str)

# Create edges based on rows
edges = list(zip(df['ctx_node'], df['rep_node']))
G.add_edges_from(edges)

# 3. Find Connected Components
components = list(nx.connected_components(G))

# Map every node to a unique Component ID
node_to_component = {}
for component_id, nodes in enumerate(components):
    for node in nodes:
        node_to_component[node] = component_id

# 4. Map the Component IDs back to the original dataframe
# (You only need to check the context node, as the reply node is guaranteed to have the same ID)
df['component_id'] = df['ctx_node'].map(node_to_component)

# 5. Split the data ensuring components are kept together
# GroupShuffleSplit ensures all rows with the same component_id stay in the same split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, val_idx = next(gss.split(df, groups=df['component_id']))

train_df = df.iloc[train_idx].drop(columns=['ctx_node', 'rep_node', 'component_id', 'split', 'group', 'is_toxic', 'label_id', 'previous_reply', 'target_reply'])
val_df = df.iloc[val_idx].drop(columns=['ctx_node', 'rep_node', 'component_id', 'split', 'group', 'is_toxic', 'label_id', 'previous_reply', 'target_reply'])

print(f"Training rows: {len(train_df)}")
print(f"Validation rows: {len(val_df)}")

Training rows: 17991
Validation rows: 1440


In [2]:
train_df

,row_id,prev_clean,target_clean,label
0,ef2a32c7af5a0a660e0c,i shared photos from today event,in this video your body draws more attention t...,Sexual Harassment / Objectification
1,c7a506deea76fe585973,i see no reason to change my opinion,hearing this statement it seems imagination di...,Mockery / Ridicule
2,5ff6cb853fa2d2809023,there is no need to take men seriously in this...,not everyone is the same people should be trea...,NaN
3,577f23b0dbafdca5c219,what is the update on today work ?,so far more details can be given later,NaN
4,ca4f30bb8940a17405a0,i only disagreed politely,i am warning you if you ask this question agai...,Threat
...,...,...,...,...
19425,9b0ae98169e569d7e4af,that supporter should be frightened into silence,in my opinion i disagree violence is not a sol...,NaN
19426,dfd4dd9d111500c10cfa,online the post says all tickets are sold out,to put it calmly it is possible but it can not...,NaN
19427,8727ba0f549d06423fae,what did you finally decide about using a new ...,giving time to a unruly person like you was my...,Insult
19428,f3ec877da8d003e55d1d,has the travel schedule been finalized ?,as far as i know the plan has not changed so f...,NaN


In [3]:
val_df

,row_id,prev_clean,target_clean,label
11,ac1884c330e2b5c38b7b,do you know what happened at the office today ...,they started shouting at me and called me a il...,NaN
33,0cc79cc911e43d0cc3d5,evidence has been published against some corru...,these people are like hyenas pouncing whenever...,Dehumanization
38,f122a994cd3b506ffb1c,if you are this negligent again about washing ...,alright then let us see what happens in the en...,NaN
56,5a6e543bb19ae91c6722,people are angry about the actions of that fra...,they are like vultures only waiting for other ...,Dehumanization
59,3368268fa6bac87ae75b,i am jewish i only answered the question,to me i do not trust you because of your jewis...,Identity Attack
...,...,...,...,...
19331,dbf5d0d83cd1f757bab9,people are angry about the actions of that fra...,they are a pack of hyenas surrounding anyone w...,Dehumanization
19338,746c8f59c00a0811a743,those fraudsters are being discussed,that ring behave like sewer insects they are d...,Dehumanization
19363,8367e0d4051ae548bcc1,what did you think after reading this status ?,these pigs should be burned alive,Dehumanization
19396,7e1f7be2dcf3aa358692,people are demanding the attackers be brought ...,their group are like a virus spreading damage ...,Dehumanization


In [6]:
train_df.rename(columns={'prev_clean': 'context', 'target_clean': 'reply'}, inplace=True)
val_df.rename(columns={'prev_clean': 'context', 'target_clean': 'reply'}, inplace=True)

In [ ]:
import pandas as pd
import numpy as np

from scipy.sparse import hstack

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import (
  VotingClassifier,
  RandomForestClassifier,
  ExtraTreesClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.svm import SVC
from sklearn.metrics import (
  accuracy_score,
  f1_score,
  classification_report,
  confusion_matrix
)

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [5]:
target_col = 'label'

In [ ]:
# Extract text columns for TF-IDF
X_context_train = train_df['context'].fillna("").astype(str)
X_reply_train = train_df['reply'].fillna("").astype(str)
y_train = train_df[target_col].fillna("nan").astype(str)

X_context_dev = val_df['context'].fillna("").astype(str)
X_reply_dev = val_df['reply'].fillna("").astype(str)
y_dev = val_df[target_col].fillna("nan").astype(str)

In [8]:
# Word-level TF-IDF
context_word = TfidfVectorizer(
  analyzer="word",
  ngram_range=(1, 2),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

reply_word = TfidfVectorizer(
  analyzer="word",
  ngram_range=(1, 2),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

In [9]:
# Character-level TF-IDF
context_char = TfidfVectorizer(
  analyzer="char_wb",
  ngram_range=(3, 5),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

reply_char = TfidfVectorizer(
  analyzer="char_wb",
  ngram_range=(3, 5),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

In [10]:
# Fit only on training data, transform both
Cw_train = context_word.fit_transform(X_context_train)
Cw_dev = context_word.transform(X_context_dev)

Rw_train = reply_word.fit_transform(X_reply_train)
Rw_dev = reply_word.transform(X_reply_dev)

Cc_train = context_char.fit_transform(X_context_train)
Cc_dev = context_char.transform(X_context_dev)

Rc_train = reply_char.fit_transform(X_reply_train)
Rc_dev = reply_char.transform(X_reply_dev)

In [14]:
# Combine all four sparse feature matrices
X_train = hstack([Cw_train, Cc_train, Rw_train, Rc_train], format="csr")
X_dev = hstack([Cw_dev, Cc_dev, Rw_dev, Rc_dev], format="csr")

print("Training feature matrix:", X_train.shape)
print("Development feature matrix:", X_dev.shape)

Training feature matrix: (17991, 36109)
Development feature matrix: (1440, 36109)


In [11]:
models = [
  ("lr", LogisticRegression(C=1.0, max_iter=1000, random_state=42)),
  ("svm", LinearSVC(C=1.0, max_iter=3000, random_state=42)),
  ("lgbm", LGBMClassifier(n_estimators=100, learning_rate=0.1, num_leaves=31, verbosity=-1, random_state=42)),
  ("xgb", XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, objective="multi:softprob", eval_metric="mlogloss", tree_method="hist", random_state=42)),
  ("catboost", CatBoostClassifier(iterations=100, depth=6, learning_rate=0.1, loss_function="MultiClass", verbose=0, random_seed=42, thread_count=-1)),
  ("rf", RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
  ("extra", ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1))
]

In [12]:
voting_model = VotingClassifier(
  estimators=models,
  voting="hard",
  n_jobs=1
)

In [25]:
print("Training ensemble... This may take a while.")
voting_model.fit(X_train, y_train)
print("Voting ensemble training completed.")

Training ensemble... This may take a while.


In [30]:
from collections import Counter
import numpy as np

predictions = []

for name, estimator in voting_model.named_estimators_.items():
    pred = np.asarray(estimator.predict(X_dev))

    # Convert (n_samples, 1) to (n_samples,)
    if pred.ndim == 2 and pred.shape[1] == 1:
      pred = pred.ravel()

    if pred.ndim != 1 or pred.shape[0] != X_dev.shape[0]:
      raise ValueError(
          f"{name} returned invalid prediction shape: {pred.shape}"
      )

    print(f"{name}: {pred.shape}")
    predictions.append(pred)

# Shape: (number_of_models, number_of_samples)
predictions = np.vstack(predictions)

# Majority vote for each sample
y_pred = np.array([
    Counter(predictions[:, i]).most_common(1)[0][0]
    for i in range(predictions.shape[1])
])

# Convert encoded integer predictions back to original class names
y_pred_labels = voting_model.le_.inverse_transform(
    np.asarray(y_pred).astype(int)
)

# Ensure the true labels have the same representation
y_dev_labels = y_dev.astype(str)

print("\n--- ENSEMBLE RESULTS ---")
print("Accuracy:", accuracy_score(y_dev_labels, y_pred_labels))
print("Macro-F1:", f1_score(
    y_dev_labels, y_pred_labels, average="macro", zero_division=0
))
print("Weighted-F1:", f1_score(
    y_dev_labels, y_pred_labels, average="weighted", zero_division=0
))

print("\nClassification Report:")
print(classification_report(
    y_dev_labels, y_pred_labels, zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_dev_labels, y_pred_labels))

lr: (1440,)
svm: (1440,)
lgbm: (1440,)
xgb: (1440,)
catboost: (1440,)
rf: (1440,)
extra: (1440,)

--- ENSEMBLE RESULTS ---
Accuracy: 0.45694444444444443
Macro-F1: 0.5448280278302281
Weighted-F1: 0.3592014761503853

Classification Report:
                                     precision    recall  f1-score   support

                     Dehumanization       0.81      0.02      0.04       690
                         Harassment       1.00      1.00      1.00         2
                        Hate Speech       0.13      0.63      0.21        27
                    Identity Attack       1.00      0.98      0.99        95
 Incitement / Encouragement of Harm       0.00      0.00      0.00         2
                             Insult       0.42      0.84      0.56       130
                 Mockery / Ridicule       1.00      0.88      0.93        81
                          Profanity       0.33      0.12      0.18        25
Sexual Harassment / Objectification       0.58      0.17      0.27  

In [ ]:
results = []

for name, model in voting_model.named_estimators_.items():
  pred = model.predict(X_dev)
  results.append({
    "Model": name,
    "Accuracy": accuracy_score(y_dev, pred),
    "Macro-F1": f1_score(y_dev, pred, average="macro", zero_division=0),
    "Weighted-F1": f1_score(y_dev, pred, average="weighted", zero_division=0)
  })

results.append({
  "Model": "Hard Voting Ensemble",
  "Accuracy": accuracy_score(y_dev, y_pred),
  "Macro-F1": f1_score(y_dev, y_pred, average="macro", zero_division=0),
  "Weighted-F1": f1_score(y_dev, y_pred, average="weighted", zero_division=0)
})

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("Macro-F1", ascending=False)

print("\n--- MODEL COMPARISON ---")
print(results_df.to_string(index=False))